# Comprendre le Ridge communal : EDA et comparaisons par blocs

Analyse **rétrospective exploratoire**, destinée à expliquer le modèle existant.
Aucune sélection de variables ni recherche d’hyperparamètres. Le test 2023–2025
reste réservé ; ses cibles ne sont pas chargées.

- EDA et ajustement : **2012–2020**, avec historique antérieur à 2012.
- Comparaisons descriptives : **2021–2022**, validation déjà consultée.
- Historique strictement antérieur à chaque date, gelé fin 2020 pour la validation.
- Géographie 2026 et versions historiques des sources non démontrées : aucune
  conclusion de performance en prévision réelle.

Préparation : environnement `swiss-votes-ml` de `environment.yml`, ou dépendances
`requirements-demo.txt` avec Jupyter/ipykernel. Exécuter toutes les cellules dans l’ordre.
Aucun téléchargement ni reconstruction des sources. Les sorties régénérables vont dans
`reports/generated/structured_eda/`.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'environment.yml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from io import BytesIO
from IPython.display import Image, display
from scripts.modeling import retrospective_demo as demo
from scripts.modeling import structured_eda as eda

OUTPUT = ROOT / 'reports/generated/structured_eda'
OUTPUT.mkdir(parents=True, exist_ok=True)

def show_figures(figures):
    for figure in figures:
        buffer = BytesIO()
        figure.savefig(buffer, format='png', dpi=100, bbox_inches='tight')
        display(Image(data=buffer.getvalue()))
        plt.close(figure)


## 1. Charger le snapshot audité et identifier les unités

Le chargeur existant vérifie les empreintes des sources et du snapshot, les clés,
les hypothèses temporelles et l’historique recomputé. Le filtre Parquet exclut les
cibles après 2022 avant leur lecture. Une ligne représente une commune × un objet ;
les lignes d’un même objet ne constituent pas des réplications indépendantes.


In [ ]:
data, source_manifest, exclusions = demo.load_inputs(ROOT)
train = data.loc[data.partition.eq('train')].copy()
validation = data.loc[data.partition.eq('validation')].copy()
partitions = data.groupby('partition').agg(
    rows=('vote_id', 'size'), objects=('vote_id', 'nunique'), dates=('vote_date', 'nunique'),
    municipalities=('municipality_id', 'nunique'), first_date=('vote_date', 'min'), last_date=('vote_date', 'max'))
display(partitions)
assert data.vote_date.max() <= demo.VALIDATION_END


## 2. Qualité et distributions : entraînement uniquement

Les codes de typologie, langue, forme juridique et domaine restent **nominaux**.
Les valeurs de partis manquantes ne sont pas des zéros. Une variable entièrement
manquante est signalée, sans suppression opportuniste ; le Ridge réutilise son
imputation existante. Une modalité dominante peut signaler une quasi-constance.

Les distributions communales comptent les observations commune × objet. Les
modalités propres à l’objet sont comptées une fois par objet. Les parts d’âge
sont liées par construction, et `80_plus` est inclus dans `65_plus`.


In [ ]:
inventory = eda.training_inventory(train)
display(inventory.sort_values('missing_share', ascending=False))
train_numeric_summary = train[demo.NUMERIC].describe().T
train_numeric_summary.to_csv(OUTPUT / 'training_numeric_summary.csv')
train[demo.NUMERIC].corr(method='spearman').to_csv(OUTPUT / 'training_correlations.csv')
display(train_numeric_summary)
show_figures(eda.quality_figures(train, OUTPUT))


## 3. Associations avec la cible : entraînement uniquement

Les scatters utilisent au plus 4 000 lignes, échantillonnées avec une graine fixe
pour la lisibilité. Les boxplots utilisent toutes les lignes d’entraînement.
Les relations avec les métadonnées d’objet utilisent **une observation par objet** :
la moyenne communale non pondérée du oui. Aucun test de significativité naïf sur
les milliers de communes d’un même objet.

Les corrélations marginales ne mesurent ni causalité ni gain prédictif conditionnel.
Leur signe ne doit pas servir à choisir les variables à supprimer.


In [ ]:
target_correlations = train[demo.NUMERIC].corrwith(train.yes_share, method='spearman').rename('spearman_yes_share')
target_correlations.to_csv(OUTPUT / 'training_target_correlations.csv')
display(target_correlations.sort_values().to_frame())
show_figures(eda.target_figures(train, OUTPUT))


## 4. Couverture et protocole des comparaisons

Ridge avec **alpha = 1** pour tous les modèles. Imputation médiane, standardisation,
encodage one-hot et indicateurs de valeurs manquantes sont ajustés sur l’entraînement.
Les prédictions sont bornées à [0, 1] avant le calcul de la MAE. Toutes les configurations
conservent les mêmes lignes d’évaluation, y compris les observations imputées.

Deux références distinguent une différence de l’expérience publiée :
- `ridge_history` reproduit son historique moyen et son indicateur d’absence.
- `ridge_history_count` ajoute le nombre d’objets historiques, également présent
  dans le modèle combiné ; c’est la **référence des comparaisons par blocs**.

Chaque bloc est ajouté à cet historique complet. Le combiné est exactement le
Ridge combiné publié. Ces ajouts ne mesurent pas l’effet du retrait d’un bloc du
modèle combiné. La couverture distingue observations réelles et valeurs imputées ;
une couverture complète peut être nulle si un parti est entièrement manquant.
Les taux par variable figurent dans l’export CSV.


In [ ]:
display(pd.DataFrame([
    {'model': name, 'numeric': ', '.join(numeric), 'categorical': ', '.join(categorical)}
    for name, (numeric, categorical) in eda.feature_sets().items()
]))
coverage = eda.source_coverage(data)
display(coverage.drop(columns='per_feature_observed_share'))


In [ ]:
fits, predictions, scores, by_object, by_date = eda.fit_comparisons(train, validation)
display(scores[['model', 'mae_pp', 'delta_vs_history_count_pp', 'mean_object_mae_pp',
                'mean_date_mae_pp', 'rows', 'clipped_rows']].sort_values('mae_pp'))
assert predictions.groupby('model').size().eq(len(validation)).all()


## 5. Coefficients et régularité des gains

Les coefficients décrivent le modèle **avant bornage**. Un coefficient numérique
correspond à un écart-type d’entraînement après imputation ; une modalité one-hot
et un indicateur d’absence sont des variables binaires, affichées séparément.
Le codage conserve toutes les modalités : les coefficients d’une modalité ne sont
pas des écarts à une catégorie de référence supprimée. Leur interprétation dépend
également de la régularisation et des autres variables corrélées. Aucun classement
global entre représentations, aucune interprétation causale.

La MAE principale donne le même poids à chaque ligne commune × objet. Les MAE
moyennes par objet et par date donnent le même poids à chaque groupe respectif.
Un delta négatif indique un gain par rapport à l’historique complet. Les figures
montrent les gains du combiné ; les CSV détaillent tous les blocs par objet/date.
**24 objets sur sept dates** ne fournissent pas une estimation finale indépendante.
L’importance par permutation est reportée.


In [ ]:
coefficients = eda.coefficient_table(fits['ridge_combined'])
coefficient_peeks = pd.concat([
    group.loc[group.coefficient_pp.abs().nlargest(8).index]
    for _, group in coefficients.groupby('representation', sort=True)
])
display(coefficient_peeks)
show_figures(eda.model_figures(coefficients, scores, by_object, by_date, OUTPUT))
display(by_date[['vote_date', 'model', 'absolute_error_pp', 'delta_vs_history_count_pp']])


## 6. Export et éléments à présenter

Les tableaux, prédictions de validation, figures PNG/SVG et manifeste conservent
les configurations et la provenance. Le notebook n’élit aucun nouveau modèle.
Un gain observé ici constitue une comparaison exploratoire sur validation réutilisée.
Les cellules suivantes fournissent des observations chiffrées, pas une preuve de
causalité ni de disponibilité des sources à l’époque des votations.


In [ ]:
manifest = eda.export_results(ROOT, OUTPUT, source_manifest, inventory, coverage, coefficients,
                              predictions, scores, by_object, by_date)
combined_objects = by_object.loc[by_object.model.eq('ridge_combined')]
combined_dates = by_date.loc[by_date.model.eq('ridge_combined')]
display(pd.Series({
    'objets avec gain du combiné vs historique complet': int(combined_objects.delta_vs_history_count_pp.lt(0).sum()),
    'objets évalués': len(combined_objects),
    'dates avec gain du combiné vs historique complet': int(combined_dates.delta_vs_history_count_pp.lt(0).sum()),
    'dates évaluées': len(combined_dates),
    'variables entièrement manquantes sur entraînement': ', '.join(inventory.index[inventory.all_missing]),
    'test évalué': manifest['test_evaluated'],
}, name='Constats descriptifs'))
print(f'Exports : {OUTPUT}')


## Lecture pour la présentation

1. Présenter l’unité commune × objet et les périodes avant les scores.
2. Décrire les manquants et redondances sans en déduire une sélection de variables.
3. Comparer l’historique complet aux ajouts de blocs sur les mêmes observations.
4. Montrer les écarts par objet/date pour rendre visibles les gains et les échecs.
5. Interpréter les coefficients avec leurs unités et leurs dépendances.
6. Rappeler la validation réutilisée, la géographie rétrospective et le test réservé.

Une comparaison où le bloc n’améliore pas l’historique ne suffit pas à déclarer
ses variables inutiles dans toutes les combinaisons. Le travail ne démontre pas
l’apport des textes de brochures ni une performance de prévision opérationnelle.
